# SigLIP2 versus DINOv2: a low-resource bean-leaf classification comparison

Reproduce **12 new runs**: two official backbones, frozen and partial fine-tuning arms,
and seeds 17, 42 and 2026. The newer checkpoint is
`google/siglip2-base-patch16-224` (2025); the comparator is
`facebook/dinov2-small` (2023). Both are publicly downloadable at pinned revisions.
SigLIP2's vision tower and pretrained image pooler supply the representation; a new
three-class head is trained. This is supervised image classification, not zero-shot
text prompting. Partial adaptation updates the final two blocks and final normalization;
SigLIP2's pretrained attention pooler remains frozen.

**Interpretation:** the original DINOv2 test results were already known when this
follow-up was planned. The comparison is explicitly exploratory. A newer competitive
model family does not establish the best model available in October 2026 or a
state-of-the-art Beans result. Model sizes, pretraining, pooling and native preprocessing
differ. The fixed learning rates are not independently optimized for each model.

**Notebook status:** supplied unexecuted. The project's saved results document local
execution; they do not establish that this notebook was executed in Google Colab.
`model_comparison_review.ipynb` reads those saved results without training.

Read the [current technical report](report/technical_report.pdf) for the comparison
findings. The [original DINOv2 report](report/archive/dinov2_study_2026-10-23/technical_report.pdf)
is retained as historical evidence. Neither the newer release date nor this notebook
implies that SigLIP2 achieves a higher score; use the completed results.


## 1. Locate the project and install the reference environment

In Colab, choose **Runtime → Change runtime type → GPU**. Upload the complete project
ZIP if this notebook was opened alone. GPU allocation is not guaranteed. The dataset
and official model downloads do not require an account token or approval.
Install before importing scientific packages; restart the session if already-imported
packages change. Python 3.11 is the tested reference version. Downloads, installation,
checkpoint loading and analysis add time beyond the measured training loops.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import copy, hashlib, json, os, shutil, subprocess, sys, zipfile

IN_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))
candidates = [Path.cwd(), Path.cwd() / "beans_vit_project",
              Path.cwd() / "outputs/beans_vit_project", Path("/content/beans_vit_project")]
PROJECT_ROOT = next((p.resolve() for p in candidates
                     if (p / "src/run_model_comparison.py").is_file()), None)
if PROJECT_ROOT is None and IN_COLAB:
    from google.colab import files
    print("Upload the complete project ZIP, containing src/ and requirements.txt.")
    uploaded = files.upload()
    archives = [Path(name) for name in uploaded if name.lower().endswith(".zip")]
    if len(archives) != 1:
        raise RuntimeError("Upload exactly one complete project ZIP.")
    target = Path("/content/beans_vit_comparison_upload").resolve()
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archives[0]) as archive:
        for entry in archive.infolist():
            destination = (target / entry.filename).resolve()
            if not destination.is_relative_to(target):
                raise ValueError("Unsafe ZIP path")
        archive.extractall(target)
    matches = [p.parent.parent for p in target.rglob("src/run_model_comparison.py")]
    if len(matches) != 1:
        raise RuntimeError("Expected one project with src/run_model_comparison.py.")
    PROJECT_ROOT = matches[0]
if PROJECT_ROOT is None:
    raise RuntimeError("Open from the full extracted project or use the Colab upload option.")
os.chdir(PROJECT_ROOT)
os.environ["HF_HOME"] = str(PROJECT_ROOT / "work/huggingface")
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
print("Project:", PROJECT_ROOT, "| Python:", sys.version)


In [ ]:
install = [sys.executable, "-m", "pip", "install", "-r", str(PROJECT_ROOT / "requirements.txt")]
if IN_COLAB:
    install += ["torchvision==0.21.0", "torchaudio==2.6.0"]
subprocess.run(install, check=True)
print("Installation finished. Restart if scientific packages were already imported.")


## 2. Check resources and create a separate reproduction directory

This workflow preserves the supplied study plan and original evidence. A fresh run ID
gives the reproduction its own plan, audit, checkpoints and results. It uses the exact
three seeds and settings in the supplied plan. Do not silently reduce runs or tune from
test scores. An intentional protocol change requires a separately documented study.
To resume after a kernel restart, replace the `RUN_ID` expression with the exact
previously printed ID. Existing audit/plan files are reused and verified, not rewritten.

CPU execution is disabled by default. Set `ALLOW_CPU=True` only as an explicit decision;
no CPU completion-time promise is made. Allow several GB for model downloads and all
selected checkpoints. Temporary Colab storage is not a backup.


In [ ]:
import torch
import numpy as np
import pandas as pd
from IPython.display import display, Image

ALLOW_CPU = False
RUN_ID = globals().get("RUN_ID", "comparison_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"))
if torch.__version__.split("+")[0] != "2.6.0" or np.__version__ != "2.2.6":
    raise RuntimeError("Imported versions differ from pins. Restart the session and run from the top.")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cpu" and not ALLOW_CPU:
    raise RuntimeError("No CUDA GPU allocated. Select a GPU runtime or explicitly set ALLOW_CPU=True.")
print("Device:", DEVICE, "| torch:", torch.__version__, "| CUDA build:", torch.version.cuda)
if DEVICE == "cuda":
    gpu = torch.cuda.get_device_properties(0)
    print("GPU:", gpu.name, "| VRAM GiB:", round(gpu.total_memory / 2**30, 2))

ORIGINAL_STUDY = PROJECT_ROOT / "results/model_comparison"
ORIGINAL_PLAN = ORIGINAL_STUDY / "study_plan.json"
ORIGINAL_TIMING = ORIGINAL_STUDY / "timing_analysis_amendment.json"
STUDY = PROJECT_ROOT / "results/reproduction" / RUN_ID
RUN_ROOT = PROJECT_ROOT / "work/comparison_runs" / RUN_ID
DATA_ROOT = PROJECT_ROOT / "work/data/beans"
IMAGE_ROOT = DATA_ROOT / "images"
AUDIT = STUDY / "data_audit"
MANIFEST = AUDIT / "manifest.csv"
PLAN_PATH = STUDY / "study_plan.json"
TIMING_PATH = STUDY / "timing_analysis_amendment.json"
assert STUDY.resolve() != ORIGINAL_STUDY.resolve()
STUDY.mkdir(parents=True, exist_ok=True)

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def run(script, *args):
    command = [sys.executable, str(PROJECT_ROOT / "src" / script), *map(str, args)]
    print("Running", script, flush=True)
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)

print("New evidence:", STUDY)
print("Checkpoint directory:", RUN_ROOT)
print("Disk free GiB:", round(shutil.disk_usage(PROJECT_ROOT).free / 2**30, 1))


## 3. Download and audit the pinned Beans dataset

Preserve the published 1,034/133/128 train/validation/test split. Download verification,
image decoding, exact file/pixel duplicate checks and pHash screening run again here.
Reuse recorded AI-assisted visual decisions only for matching pairs from this exact
release. This does not validate agricultural labels or establish farm independence.
The original integrity audit inspected test images before modelling; the follow-up
also starts with previous test-result exposure, which remains disclosed.


In [ ]:
if PLAN_PATH.exists() or (STUDY / "execution_identity.json").exists():
    existing_plan = read_json(PLAN_PATH)
    assert sha256(MANIFEST) == existing_plan["dataset"]["manifest_sha256"]
    print("Reusing the existing immutable manifest:", MANIFEST)
else:
    AUDIT.mkdir(parents=True, exist_ok=True)
    for name in ["review_decisions.csv", "visual_review_record.json"]:
        source = PROJECT_ROOT / "results/data_audit" / name
        if source.is_file():
            shutil.copy2(source, AUDIT / name)
    run("prepare_data.py", "--data-root", DATA_ROOT, "--output-dir", AUDIT,
        "--review-decisions", AUDIT / "review_decisions.csv")
audit = read_json(AUDIT / "audit_summary.json")
assert audit["dataset_revision"] == "27aa014ce09b193e1a6f58112d4a66e0eddb69c5"
assert audit["included_counts"] == {"train": 1034, "validation": 133, "test": 128}
assert not audit["corrupt_images"]
unresolved = {key: count for key, count in audit["visual_review_status"].items()
              if key not in {"distinct", "confirmed_duplicate"} and count}
if unresolved:
    raise RuntimeError(f"Inspect unresolved duplicate candidates before proceeding: {unresolved}")
print(json.dumps({key: audit[key] for key in ["included_counts", "corrupt_images",
                  "near_duplicate_candidate_pairs", "visual_review_status"]}, indent=2))
display(Image(filename=str(AUDIT / "dataset_examples.png"), width=850))


## 4. Record the reproduction plan before the pilot and full training

Absolute paths in the regenerated manifest change its byte hash. Copy the original
plan into this reproduction, replace that manifest path/hash, record the new creation
time and the original plan's checksum, and keep every experimental setting unchanged.
The original plan and original results remain intact. The historical test-exposure
limitation is retained; a reproduction does not undo previous knowledge of the test set.
Record the descriptive timing policy before execution as well, linking the original
timing amendment. Its 809-second observation belongs to the original run; it is not
asserted to have occurred in your reproduction.


In [ ]:
original_hash = sha256(ORIGINAL_PLAN)
original = read_json(ORIGINAL_PLAN)
original_timing_hash = sha256(ORIGINAL_TIMING)
original_timing = read_json(ORIGINAL_TIMING)
if TIMING_PATH.exists():
    timing_policy = read_json(TIMING_PATH)
    assert timing_policy["original_amendment_sha256"] == original_timing_hash
    assert timing_policy["before_new_test_evaluation"] is True
    assert timing_policy["raw_times_preserved"] is True
else:
    if (STUDY / "execution_identity.json").exists():
        raise RuntimeError("Execution began without the timing policy; do not retrospectively claim a pre-execution record.")
    observed_test_files = [path for pattern in ["metrics_test*.json", "predictions_test*.csv"]
                           for path in RUN_ROOT.rglob(pattern)]
    assert not observed_test_files, "Cannot record a pre-evaluation policy after test outputs exist."
    timing_policy = {key: original_timing[key] for key in [
        "training_changes", "checkpoint_selection_changes", "test_evaluation_changes",
        "raw_times_preserved", "existing_metrics_retained", "additional_descriptive_statistics",
        "quantile_method", "cost_figure_metric", "exclusions_or_corrections", "interpretation"]}
    timing_policy.update({
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "status": "Timing summary policy carried forward before this reproduction's execution",
        "before_new_test_evaluation": True,
        "original_dinov2_test_results_previously_known": True,
        "observed_new_test_artifact_count": 0,
        "original_amendment": "results/model_comparison/timing_analysis_amendment.json",
        "original_amendment_sha256": original_timing_hash,
        "original_amendment_created_utc": original_timing["created_utc"],
        "motivation": "Apply the original descriptive timing policy to all reproduction epochs; the original unusually long epoch is historical and is not asserted for this runtime.",
    })
    with TIMING_PATH.open("x", encoding="utf-8") as stream:
        json.dump(timing_policy, stream, indent=2)
        stream.write("\n")
if PLAN_PATH.exists():
    plan = read_json(PLAN_PATH)
    assert plan["reproduction"]["original_plan_sha256"] == original_hash
    assert plan["dataset"]["manifest_sha256"] == sha256(MANIFEST)
    assert plan["reproduction"]["run_id"] == RUN_ID
    print("Reusing the existing plan without changing it.")
else:
    if (STUDY / "execution_identity.json").exists():
        raise RuntimeError("Execution began but its plan is missing; do not reconstruct it retrospectively.")
    plan = copy.deepcopy(original)
    plan["created_utc"] = datetime.now(timezone.utc).isoformat()
    plan["study_status"] = "Reproduction of the fixed exploratory follow-up; original test exposure remains disclosed"
    plan["dataset"]["manifest"] = str(MANIFEST.resolve())
    plan["dataset"]["manifest_sha256"] = sha256(MANIFEST)
    plan["reproduction"] = {
        "run_id": RUN_ID,
        "original_plan": "results/model_comparison/study_plan.json",
        "original_plan_sha256": original_hash,
        "original_plan_created_utc": original["created_utc"],
        "experimental_settings_changed": False,
        "regenerated_manifest_has_runtime_specific_absolute_paths": True,
        "prior_test_exposure_retained": True,
        "timing_policy_sha256": sha256(TIMING_PATH),
    }
    with PLAN_PATH.open("x", encoding="utf-8") as stream:
        json.dump(plan, stream, indent=2)
        stream.write("\n")
assert sha256(ORIGINAL_PLAN) == original_hash
assert plan["settings"] == original["settings"] and plan["models"] == original["models"]
assert plan["reproduction"]["timing_policy_sha256"] == sha256(TIMING_PATH)
assert sha256(ORIGINAL_TIMING) == original_timing_hash
print(json.dumps({"models": plan["models"], "settings": plan["settings"]}, indent=2))
print("Recorded reproduction plan:", PLAN_PATH, "| SHA256:", sha256(PLAN_PATH))

def phase(name):
    run("run_model_comparison.py", "--manifest", MANIFEST, "--data-root", IMAGE_ROOT,
        "--run-root", RUN_ROOT, "--out", STUDY, "--study-plan", PLAN_PATH,
        "--device", DEVICE, "--phase", name, "--resume")


## 5. Run full-epoch pilots for both models and both arms

Each pilot checks gradients, frozen parameters and checkpoint round trips, and
benchmarks one complete training epoch. Full experiments restart from the pinned
pretrained weights. The scripts download official checkpoints automatically and use
their pinned revisions; no remote executable model code is required. No test
prediction is requested by this phase. Check actual memory and timing before proceeding.


In [ ]:
phase("pilot")
pilots = []
for model in plan["models"]:
    records = read_json(STUDY / model / "pilot_report.json")
    assert len(records) == 2 and all(item["status"] == "passed" for item in records)
    for item in records:
        pilots.append({"model": model, "arm": item["arm"],
                       "one_training_epoch_seconds": item["training"]["seconds"],
                       "peak_cuda_allocated_bytes": item["peak_cuda_allocated_bytes"]})
display(pd.DataFrame(pilots))
training_minutes = sum(row["one_training_epoch_seconds"] for row in pilots) * plan["settings"]["epochs"] * len(plan["settings"]["seeds"]) / 60
print("Full-epoch-budget training-only estimate in minutes:", round(training_minutes, 1))
print("This excludes validation, downloads, loading, checkpoint I/O and analysis.")


## 6. Train all 12 runs with validation-only checkpoint selection

Images receive the model's recorded full-image resizing and normalization. Augmentation
randomness and image order are matched across backbones by seed, epoch and image ID.
Classifier initialization is paired within a backbone; embedding dimensions differ
across backbones. AdamW settings, batch size, epoch limit and stopping rule are fixed.
Select by highest validation macro-F1, then lower loss, then earlier epoch. Only a strict
F1 improvement resets patience. The implementation permits platform-specific
nondeterministic CUDA operations with a warning; bitwise identical reruns are not promised.

Completed matching runs may be reused with `--resume`; incomplete runs are never
overwritten automatically. Keep temporary runtime checkpoints if resumption is needed.


In [ ]:
phase("train")
summaries = []
for model in plan["models"]:
    for arm in plan["settings"]["arms"]:
        for seed in plan["settings"]["seeds"]:
            folder = RUN_ROOT / model / "runs" / arm / f"seed_{seed}"
            summary = read_json(folder / "summary.json")
            assert (folder / "best.pt").is_file()
            summaries.append({"model": model, **{key: summary[key] for key in
                ["arm", "seed", "best_epoch", "epochs_completed", "best_validation_macro_f1", "training_seconds"]}})
assert len(summaries) == 12
display(pd.DataFrame(summaries))
print("Training complete. The table shows validation selection; this phase did not request test inference.")


## 7. Lock the selected checkpoints and evaluate the existing test split

The driver requires every planned run, verifies configurations and hashes, and creates
both model locks before new test inference. The validator checks membership and hashes
before each evaluation. The lock records that this is an exploratory reuse of a test
set evaluated in the earlier study, not a fresh confirmatory holdout.

Evaluate clean test images for all 12 selected checkpoints. Seed 17 additionally uses
the already-fixed blur radius 1 and brightness ×0.7 conditions. These are paired
synthetic sensitivity checks, not new independent field observations. Do not use their
scores for more tuning or replace checkpoints after seeing test results.


In [ ]:
phase("evaluate")
print(json.dumps(read_json(STUDY / "evaluation_completion.json"), indent=2))
for model in plan["models"]:
    lock = read_json(STUDY / model / "protocol.lock.json")
    assert lock["original_dinov2_test_results_known_before_followup"] is True
    assert lock["new_followup_test_predictions_evaluated_before_lock"] is False
    print(model, "lock:", sha256(STUDY / model / "protocol.lock.json"))


## 8. Analyse metrics, uncertainty, cost and overfitting

The analysis recomputes metrics from saved predictions and produces all-seed summaries,
class metrics, confusion matrices, real sample errors and synthetic sensitivity figures.
The primary model difference is partial SigLIP2 minus partial DINOv2. Seed 17 uses
2,000 paired class-stratified bootstrap resamples with seed 20261023. This descriptive
interval is conditional on the reused test sample and reference pair; it is not an
interval for the three-seed mean or an independent superiority confirmation.

**Timing analysis amendment:** an unusually long epoch was observed during the original
follow-up's training, before its test evaluation. The amended analysis adds median,
quartiles and range of observed epoch times while retaining every raw epoch duration
and total. These describe the recorded hardware session; they are not a controlled
throughput benchmark or proof of the slowdown's cause. Your reproduction may have
different timing variation. Compare total observed training cost as well as medians.
The completed original follow-up also contains a 184.36-second SigLIP2 partial epoch,
as well as the 809.10-second frozen epoch that motivated the amendment. Both remain
in all timing summaries; neither has an established cause.

Clean training and validation inference scores each unchanged selected checkpoint with
no augmentation or parameter updates. Saved test metrics provide context. All-seed
learning curves reveal late deterioration; a small clean train-validation gap cannot
prove absence of overfitting, leakage or distribution shift. Online augmented training
loss and fixed-checkpoint clean loss are different evaluations.


In [ ]:
phase("analyse")
completion = read_json(STUDY / "completion.json")
assert completion["status"] == "completed" and completion["run_count"] == 12
summary = pd.read_csv(STUDY / "combined_summary.csv")
display(summary)
timing_columns = ["model", "arm", "pooled_epochs", "pooled_epoch_median_seconds",
                  "pooled_epoch_q1_seconds", "pooled_epoch_q3_seconds", "pooled_epoch_iqr_seconds",
                  "pooled_epoch_min_seconds", "pooled_epoch_max_seconds", "pooled_training_seconds",
                  "pooled_training_seconds_per_epoch", "training_seconds_mean"]
display(summary[timing_columns])
display(pd.read_csv(STUDY / "paired_model_differences.csv"))
analysis = read_json(STUDY / "comparison_analysis.json")
print(json.dumps(analysis["primary_comparison"], indent=2))
print("Timing interpretation:", analysis["timing_analysis"]["interpretation"])
for stem in ["performance_cost", "partial_confusion", "reference_learning_curves",
             "siglip_sample_predictions", "model_generalisation"]:
    display(Image(filename=str(STUDY / "figures" / (stem + ".png")), width=1000))
for model in plan["models"]:
    print(model, "selected-checkpoint clean gaps and trajectory diagnostics")
    display(pd.read_csv(STUDY / model / "overfitting/summary.csv"))
    display(pd.read_csv(STUDY / model / "overfitting/history_diagnostics.csv"))
print("Interpretation limits:", *analysis["interpretation_limits"], sep="\n- ")


## 9. Export evidence before the runtime ends

Export audit files, plans, locks, configurations, histories, predictions, metrics,
figures and the environment listing. Weights, raw images and caches are excluded.
Save `RUN_ROOT` separately if you need checkpoint inference without retraining.
The bundle documents this reproduction; it must not replace the original reported
evidence without updating the report and identifying the new run.


In [ ]:
environment = subprocess.run([sys.executable, "-m", "pip", "freeze"],
                             check=True, capture_output=True, text=True).stdout
(STUDY / "environment.txt").write_text(environment, encoding="utf-8")
export = PROJECT_ROOT / "work" / (RUN_ID + "_evidence.zip")
export.parent.mkdir(parents=True, exist_ok=True)
allowed = {".json", ".csv", ".png", ".pdf", ".md", ".txt"}
with zipfile.ZipFile(export, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(STUDY.rglob("*")):
        if path.is_file() and path.suffix.lower() in allowed:
            archive.write(path, arcname=str(path.relative_to(STUDY)))
print("Saved:", export, "| MiB:", round(export.stat().st_size / 2**20, 2))
if IN_COLAB:
    from google.colab import files
    files.download(str(export))


## References, attribution and responsible interpretation

- [Makerere Beans source](https://github.com/AI-Lab-Makerere/ibean) and
  [pinned distribution](https://huggingface.co/datasets/AI-Lab-Makerere/beans/tree/27aa014ce09b193e1a6f58112d4a66e0eddb69c5).
  MIT license; retain `THIRD_PARTY_NOTICES.md` when redistributing.
- [DINOv2 paper](https://arxiv.org/abs/2304.07193) and
  [official pinned DINOv2-Small checkpoint](https://huggingface.co/facebook/dinov2-small/tree/ed25f3a31f01632728cabb09d1542f84ab7b0056).
- [SigLIP 2 paper](https://arxiv.org/abs/2502.14786) and
  [official pinned SigLIP2 checkpoint](https://huggingface.co/google/siglip2-base-patch16-224/tree/75de2d55ec2d0b4efc50b3e9ad70dba96a7b2fa2).
- [Colab resource and runtime FAQ](https://research.google.com/colaboratory/faq.html).

The dataset has only three categories and limited geography. Unknown farm/plant grouping,
unknown pretraining overlap and uncalibrated confidence constrain deployment claims.
Errors can produce missed disease or unnecessary treatment; this benchmark does not
validate field diagnosis. Disclose AI assistance according to course rules and verify
the agricultural interpretation yourself. No automated review guarantees a grade.
